In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Download text helper files
nltk.download('punkt')
nltk.download('stopwords')

# Create a folder to save results
os.makedirs('results', exist_ok=True)

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower() # 1. Lowercase
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text) # 2. Remove special characters
    tokens = word_tokenize(text) # 3. Split into individual words
    filtered_tokens = [w for w in tokens if w not in stop_words] # 4. Remove boring stopwords (e.g., "the", "is")
    return " ".join(filtered_tokens)

df['cleaned_text'] = df['text'].apply(clean_text)
print("Before Clean:", df['text'].iloc[0])
print("After Clean :", df['cleaned_text'].iloc[0])

In [ ]:
# Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(df['cleaned_text'], df['label'], test_size=0.2, random_state=42)

# Convert text to numbers using TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Train the Baseline Model
baseline_model = LogisticRegression()
baseline_model.fit(X_train_tfidf, y_train)

# Check results
y_pred = baseline_model.predict(X_test_tfidf)
print("Baseline Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
max_words = 10000
max_len = 50

# Convert words to consistent number sequences
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['cleaned_text'])
X_seq = tokenizer.texts_to_sequences(df['cleaned_text'])
X_padded = pad_sequences(X_seq, maxlen=max_len, padding='post', truncating='post')

# Split sequential data
X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(X_padded, df['label'], test_size=0.2, random_state=42)

# Build LSTM Architecture
model = Sequential([
    Embedding(input_dim=max_words, output_dim=64, input_length=max_len),
    LSTM(64),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Outputs a probability between 0 and 1
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Train the network
history = model.fit(X_train_seq, y_train_seq, epochs=3, batch_size=32, validation_data=(X_test_seq, y_test_seq))

In [ ]:
# Save Accuracy Plot
plt.figure()
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('LSTM Performance')
plt.legend()
plt.savefig('results/model_evaluation.png')

# Save Sample Predictions
predictions = model.predict(X_padded[:3])
with open('results/sample_predictions.txt', 'w') as f:
    f.write("Sample Evaluation Outputs:\n")
    for i, pred in enumerate(predictions):
        f.write(f"Text: {df['text'].iloc[i]} -> Score: {pred[0]:.4f}\n")
print("Results saved successfully inside the 'results/' folder!")